In [3]:
pip install pandas

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 24.8 MB/s  0:00:00eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd


def load_and_explode_tsv(path: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Reads a TSV where TAGS may span multiple tab-separated cells (variable number of columns),
    then returns:
      - df: normalized dataframe with TAGS as a list
      - df_exploded: one row per tag (exploded), with parsed tag_key/tag_value
    """
    # Read raw lines because TAGS can have multiple tab-separated values (variable-width rows)
    with open(path, "r", encoding="utf-8") as f:
        lines = [ln.rstrip("\n") for ln in f if ln.strip()]

    header = lines[0].split("\t")
    base_cols = header[:5]  # TRACK_ID, ARTIST_ID, ALBUM_ID, PATH, DURATION

    rows: list[dict] = []
    for ln in lines[1:]:
        parts = ln.split("\t")
        if len(parts) < 5:
            continue

        row = {
            "TRACK_ID": parts[0],
            "ARTIST_ID": parts[1],
            "ALBUM_ID": parts[2],
            "PATH": parts[3],
            "DURATION": float(parts[4]) if parts[4] != "" else None,
            "TAGS": [p for p in parts[5:] if p],  # all remaining cells are tags
        }
        rows.append(row)

    df = pd.DataFrame(rows)

    df_exploded = df.explode("TAGS", ignore_index=True)

    # Optional: parse "genre---metal" into tag_key="genre", tag_value="metal"
    kv = df_exploded["TAGS"].astype("string").str.split("---", n=1, expand=True)
    df_exploded["tag_key"] = kv[0]
    df_exploded["tag_value"] = kv[1]

    return df, df_exploded



df, df_exploded = load_and_explode_tsv("assets/raw_30s.tsv")

print("DF (list tags):")
print(df.head(10))

print("\nDF exploded (1 row per tag):")
print(df_exploded.head(20))

FileNotFoundError: [Errno 2] No such file or directory: 'tracks.tsv'